# Paper figures — specificity (Analisi 1) & mass (Analisi 2)
Rigenera le figure dai CSV. English, no title, viridis; PNG 300dpi + PDF vettoriale. Cambia `SCORE`/`KIND`/`FS`.

## Config & helpers

In [ ]:
import os, re
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

OUT = '../grad/xai_outputs'
FIG = os.path.join(OUT, 'paper_figures'); os.makedirs(FIG, exist_ok=True)

SCORE, KIND, VERSIONS, CMAP = 'combined_score', 'TP', ['top3', 'top5'], 'viridis'
FS, MK, W, H = 20, 150, 0.46, 0.40            # font, marker, larghezza/altezza per cella

plt.rcParams.update({'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': FS, 'axes.labelsize': FS+1, 'xtick.labelsize': FS-2, 'ytick.labelsize': FS-2,
    'legend.fontsize': FS-2, 'legend.title_fontsize': FS-1, 'axes.linewidth': 0.7,
    'xtick.major.width': 0.7, 'ytick.major.width': 0.7, 'pdf.fonttype': 42, 'ps.fonttype': 42,
    'savefig.dpi': 300, 'figure.dpi': 120})

SPEC = {'jaccard':    dict(csv='jaccard_uniqueness_ALL_top5_absC.csv', contrast={'TP':'TPvsTN','pred':'predvsTN'}, has_K=True,  mcol='jaccard'),
        'uniqueness': dict(csv='jaccard_uniqueness_ALL_top5_absC.csv', contrast={'TP':'TPvsTN','pred':'predvsTN'}, has_K=True,  mcol='uniqueness'),
        'spearman':   dict(csv='spearman_specificity_ALL_top5_absC.csv', contrast={'TP':'TP<TN','pred':'pred<TN'}, has_K=False, mcol=None)}
METRICS = ['jaccard', 'uniqueness']           # Spearman escluso dalle figure
KORDER = ['3','5','10','20','50','100','all']
KMARK  = dict(zip(KORDER, ['P','X','o','s','^','D','o']))
_cache = {}

def _load(csv):
    _cache.setdefault(csv, pd.read_csv(os.path.join(OUT, csv))); return _cache[csv]

def get_metric(metric, version):
    sp = SPEC[metric]; d = _load(sp['csv'])
    d = d[(d['version']==version) & (d['score']==SCORE) & (d['contrast']==sp['contrast'][KIND])].copy()
    if sp['mcol']: d = d[d['metric']==metric]
    d['MWU_p'] = pd.to_numeric(d['MWU_p'], errors='coerce')
    d = d[d['MWU_p'].round(2) <= 0.05].copy()
    d['negp'] = -np.log10(d['MWU_p'].clip(lower=1e-6))
    d['Klab'] = 'all' if not sp['has_K'] else d['K'].astype(int).astype(str)
    return d[['tissue','target','Klab','negp']]

def _natkey(s):
    m = re.match(r'SBS(\d+)(.*)', s); return (int(m.group(1)), m.group(2)) if m else (999, s)

def panel(ax, d, tums, sigs, vmax, spread=True):
    xi = {t:i for i,t in enumerate(tums)}; yi = {s:i for i,s in enumerate(sigs)}
    klabs = [k for k in KORDER if k in set(d['Klab'])]
    for k in klabs:
        s2 = d[d['Klab']==k]
        off = (klabs.index(k)-(len(klabs)-1)/2)*(0.62/max(len(klabs),1)) if spread else 0
        ax.scatter(s2['tissue'].str.upper().map(xi)+off, s2['target'].map(yi), c=s2['negp'],
                   cmap=CMAP, vmin=0, vmax=vmax, marker=KMARK[k], s=MK, edgecolor='0.25', linewidth=0.5, zorder=3)
    ax.set_xticks(range(len(tums))); ax.set_xticklabels(tums, rotation=90)
    ax.set_yticks(range(len(sigs))); ax.set_yticklabels(sigs)
    ax.set_xlim(-0.7, len(tums)-0.3); ax.set_ylim(-0.8, len(sigs)-0.2)
    ax.grid(color='0.93', lw=0.4, zorder=0); ax.tick_params(length=3)
    for s_ in ('top','right'): ax.spines[s_].set_visible(False)
    return klabs

def cbar(fig, ax, vmax, frac=0.02):
    cb = fig.colorbar(plt.cm.ScalarMappable(cmap=CMAP, norm=plt.Normalize(0, vmax)), ax=ax, fraction=frac, pad=0.01)
    cb.set_label(r'$-\log_{10}(p)$', fontsize=FS+1); cb.ax.tick_params(labelsize=FS-2)

def klegend(ax, klabs, anchor=(1.02, 1.0)):
    ks = [k for k in KORDER if k in klabs and k != 'all']
    if ks: ax.legend(handles=[Line2D([],[],marker=KMARK[k],ls='',color='0.35',markersize=9,label=f'K = {k}') for k in ks],
                     title='top-K patches', loc='upper left', bbox_to_anchor=anchor, frameon=False)

def save(fig, name):
    for ext in ('png','pdf'): fig.savefig(os.path.join(FIG, f'{name}.{ext}'), bbox_inches='tight', facecolor='white')
    print('  saved', name)

print('config:', SCORE, KIND, VERSIONS, '| FS =', FS)

## Fig 1 — Jaccard + Uniqueness affiancate (no title)

In [ ]:
for version in VERSIONS:
    ds = {m: get_metric(m, version) for m in METRICS}
    alld = pd.concat(ds.values(), ignore_index=True)
    if alld.empty: print(f'[{version}] vuoto'); continue
    tums = sorted(alld['tissue'].str.upper().unique()); sigs = sorted(alld['target'].unique(), key=_natkey)
    vmax = float(alld['negp'].max()); n = len(METRICS)
    fig, axes = plt.subplots(1, n, figsize=(W*len(tums)*n+3.5, H*len(sigs)+2.2), sharey=True, gridspec_kw={'wspace':0.10})
    axs = np.atleast_1d(axes); kseen = []
    for ax, m in zip(axs, METRICS):
        kseen += [k for k in panel(ax, ds[m], tums, sigs, vmax, SPEC[m]['has_K']) if k not in kseen]
        ax.set_xlabel('Tumor type')
    axs[0].set_ylabel('Mutational signature')
    cbar(fig, axs.tolist(), vmax, 0.012); klegend(axs[-1], kseen, (1.18, 1.0))
    save(fig, f'fig1_specificity_{SCORE}_{KIND}_{version}'); plt.show()

## Fig 2 — griglie singole per metrica

In [ ]:
for version in VERSIONS:
    for m in METRICS:
        d = get_metric(m, version)
        if d.empty: print(f'[{version}] {m}: vuoto'); continue
        tums = sorted(d['tissue'].str.upper().unique()); sigs = sorted(d['target'].unique(), key=_natkey)
        vmax = float(d['negp'].max())
        fig, ax = plt.subplots(figsize=(W*len(tums)+3.2, H*len(sigs)+2.2))
        ks = panel(ax, d, tums, sigs, vmax, SPEC[m]['has_K'])
        ax.set_xlabel('Tumor type'); ax.set_ylabel('Mutational signature')
        cbar(fig, ax, vmax); klegend(ax, ks)
        save(fig, f'fig2_{m}_{SCORE}_{KIND}_{version}'); plt.show()

## Fig 3 — massa v_norm (Analisi 2)

In [ ]:
VN, vcon = 'vnorm_robustness_ALL_top5_absC.csv', {'TP':'TP>TN','pred':'pred>TN'}[KIND]
for version in VERSIONS:
    d = _load(VN)
    d = d[(d['version']==version) & (d['score']==SCORE) & (d['contrast']==vcon)].copy()
    d['MWU_p'] = pd.to_numeric(d['MWU_p'], errors='coerce')
    d = d[d['MWU_p'].round(2) <= 0.05].copy()
    if d.empty: print(f'[{version}] v_norm: vuoto'); continue
    d['negp'] = -np.log10(d['MWU_p'].clip(lower=1e-6)); d['Klab'] = d['K'].astype(int).astype(str)
    tums = sorted(d['tissue'].str.upper().unique()); sigs = sorted(d['target'].unique(), key=_natkey)
    vmax = float(d['negp'].max())
    fig, ax = plt.subplots(figsize=(W*len(tums)+3.2, H*len(sigs)+2.2))
    ks = panel(ax, d[['tissue','target','Klab','negp']], tums, sigs, vmax, True)
    ax.set_xlabel('Tumor type'); ax.set_ylabel('Mutational signature')
    cbar(fig, ax, vmax); klegend(ax, ks)
    save(fig, f'fig3_mass_vnorm_{SCORE}_{KIND}_{version}'); plt.show()